# 01 - Data Scraping

**Purpose:** Scrape criminal court decisions from shuukh.mn (Criminal Court of First Instance).

| | |
|---|---|
| **Inputs** | None (fetches from web) |
| **Outputs** | `data/pilot/*.html` (500 pilot cases) |
| **Runtime** | ~2 hours (500 cases at ~15s each with rate limiting) |

## Steps
1. Explore API structure
2. Enumerate available case IDs
3. Scrape pilot sample (500 cases)
4. Verify downloaded files

In [ ]:
import sys
sys.path.insert(0, '../src')

import requests
from pathlib import Path
from bs4 import BeautifulSoup
import json
import re

from scraper import (
    get_case_list_page,
    get_case_detail,
    enumerate_sampled_pages,
    scrape_cases,
    BASE_URL, HEADERS,
)

## 1. Explore API Structure

shuukh.mn exposes case listings via an AJAX endpoint that returns JSON with HTML fragments.

**Endpoint:** `GET https://shuukh.mn/site/case_ajax`

**Key parameters:**
- `court_cat=2` — Criminal cases (first instance)
- `page` — Pagination (20 cases per page)
- `daterange` — Date filter
- `bb=1` — Required flag

Individual case pages: `GET https://shuukh.mn/single_case/{case_id}`

In [ ]:
# Fetch first page to see total count and structure
result = get_case_list_page(page=1)
print(f"Total cases: {result['count']}")
print(f"Cases on page 1: {len(result['case_ids'])}")
print(f"Sample IDs: {result['case_ids'][:5]}")
print(f"Has more pages: {result['has_more']}")

In [ ]:
# Examine a single case page
sample_id = result['case_ids'][0]
html = get_case_detail(sample_id)
soup = BeautifulSoup(html, 'html.parser')

# Show the metadata table
table = soup.find('table')
if table:
    for row in table.find_all('tr'):
        cells = row.find_all(['th', 'td'])
        if len(cells) >= 2:
            print(f"{cells[0].get_text(strip=True):>25s}: {cells[1].get_text(strip=True)[:80]}")

In [ ]:
# Show first 500 chars of case text to see structure
text = soup.get_text()
print(text[:500])

## 2. Enumerate Case IDs

For the pilot, we sample random pages rather than crawling all pages. This is faster and more polite to the server.

The `enumerate_sampled_pages()` function:
1. Gets total count from page 1
2. Randomly selects ~40 pages
3. Extracts case IDs from each page
4. Samples 500 unique IDs

In [ ]:
# Load existing pilot IDs (already scraped)
pilot_ids_path = Path('../data/pilot/pilot_ids.json')
pilot_ids = json.loads(pilot_ids_path.read_text())
print(f"Pilot IDs: {len(pilot_ids)}")
print(f"First 10: {pilot_ids[:10]}")
print(f"ID range: {min(pilot_ids)} - {max(pilot_ids)}")

## 3. Pilot Scrape (500 cases)

The scraper downloads each case's HTML with polite rate limiting (1.5-3s between requests).

```python
# How the pilot was scraped (already done):
# python src/scraper.py --pilot 500 --seed 42
# 
# Or equivalently:
# pilot_ids = enumerate_sampled_pages(n_cases=500, seed=42)
# results = scrape_cases(pilot_ids, Path('data/pilot'))
```

In [ ]:
# Verify downloaded files
pilot_dir = Path('../data/pilot')
html_files = sorted(pilot_dir.glob('*.html'))
print(f"HTML files downloaded: {len(html_files)}")

# Check file sizes
sizes = [f.stat().st_size for f in html_files]
print(f"File size range: {min(sizes):,} - {max(sizes):,} bytes")
print(f"Mean file size: {sum(sizes)/len(sizes):,.0f} bytes")
print(f"Total data: {sum(sizes)/1024/1024:.1f} MB")

In [ ]:
# Show structure of a sample HTML file
sample_file = html_files[0]
sample_html = sample_file.read_text(encoding='utf-8')
sample_soup = BeautifulSoup(sample_html, 'html.parser')

print(f"Case ID: {sample_file.stem}")
print(f"HTML length: {len(sample_html):,} chars")
print()

# Key sections visible in the HTML
text = sample_soup.get_text()
for section in ['биеийн байцаалт', 'ТОДОРХОЙЛОХ', 'ТОГТООХ']:
    idx = text.find(section)
    if idx >= 0:
        print(f"Section '{section}' at char {idx}")
    else:
        print(f"Section '{section}' not found")